In [10]:
from google.colab import userdata

import requests
import pandas as pd
import sqlite3

TMDB_API_KEY = userdata.get('TMDB_API_KEY')



# Correct the Url to be just the base endpoint, and let params handle the query string.
Url = "https://api.themoviedb.org/3/movie/top_rated"

params = {
    "api_key": TMDB_API_KEY,
    "sort_by": "popularity.desc",
    "page": 1       # Page 1 gives 20 movies
}

response = requests.get(Url, params=params)
data = response.json()

# Check if 'results' key exists before trying to access it
if 'results' in data:
    movies = pd.DataFrame(data["results"])
    print(movies.head())
else:
    print(f"API Error: {data.get('status_message', 'Unknown error')}")
    movies = pd.DataFrame() # Initialize an empty DataFrame if there's an error


   adult                     backdrop_path        genre_ids   id  \
0  False  /zfbjgQE1uSd9wiPTX4VzsLi0rGG.jpg         [18, 80]  278   
1  False  /tSPT36ZKlP2WVHJLM4cQPLSzv3b.jpg         [18, 80]  238   
2  False  /kGzFbGhp99zva6oZODW5atUtnqi.jpg         [18, 80]  240   
3  False  /zb6fM1CX41D9rF9hdgclu0peUmy.jpg  [18, 36, 10752]  424   
4  False  /w4bTBXcqXc2TUyS5Fc4h67uWbPn.jpg             [18]  389   

  original_language            original_title  \
0                en  The Shawshank Redemption   
1                en             The Godfather   
2                en     The Godfather Part II   
3                en          Schindler's List   
4                en              12 Angry Men   

                                            overview  popularity  \
0  Imprisoned in the 1940s for the double murder ...     54.4910   
1  Spanning the years 1945 to 1955, a chronicle o...     40.8372   
2  In the continuing saga of the Corleone crime f...     22.8883   
3  The true story of how

In [13]:
import json

# Convert 'genre_ids' from list to JSON string
movies['genre_ids'] = movies['genre_ids'].apply(json.dumps)

# Create SQLite DB connection
conn = sqlite3.connect("movies.db")

# Save DataFrame to SQLite table
movies.to_sql("movies", conn, if_exists="replace", index=False)

conn.close()


In [14]:

conn = sqlite3.connect("movies.db")
df = pd.read_sql("SELECT * FROM movies", conn)
conn.close()

df.head()


,adult,backdrop_path,genre_ids,id,original_language,original_title,overview,popularity,poster_path,release_date,title,video,vote_average,vote_count
0,0,/zfbjgQE1uSd9wiPTX4VzsLi0rGG.jpg,"""[18, 80]""",278,en,The Shawshank Redemption,Imprisoned in the 1940s for the double murder ...,54.4910,/9cqNxx0GxF0bflZmeSMuL5tnGzr.jpg,1994-09-23,The Shawshank Redemption,0,8.717,29963
1,0,/tSPT36ZKlP2WVHJLM4cQPLSzv3b.jpg,"""[18, 80]""",238,en,The Godfather,"Spanning the years 1945 to 1955, a chronicle o...",40.8372,/3bhkrj58Vtu7enYsRolD1fZdja1.jpg,1972-03-14,The Godfather,0,8.687,22634
2,0,/kGzFbGhp99zva6oZODW5atUtnqi.jpg,"""[18, 80]""",240,en,The Godfather Part II,In the continuing saga of the Corleone crime f...,22.8883,/hek3koDUyRQk7FIhPXsa6mT2Zc3.jpg,1974-12-20,The Godfather Part II,0,8.572,13707
3,0,/zb6fM1CX41D9rF9hdgclu0peUmy.jpg,"""[18, 36, 10752]""",424,en,Schindler's List,The true story of how businessman Oskar Schind...,21.9876,/sF1U4EUQS8YHUYjNl3pMGNIQyr0.jpg,1993-12-15,Schindler's List,0,8.567,17227
4,0,/w4bTBXcqXc2TUyS5Fc4h67uWbPn.jpg,"""[18]""",389,en,12 Angry Men,The defense and the prosecution have rested an...,17.1216,/2QXLVh32JKaWTjFJU3n8aIxRK9P.jpg,1957-04-10,12 Angry Men,0,8.556,9828


In [16]:
df.describe()

,adult,id,popularity,video,vote_average,vote_count
count,20.0,2.000000e+01,20.000000,20.0,20.000000,20.00000
mean,0.0,1.121952e+05,24.996700,0.0,8.520850,18104.80000
std,0.0,2.860420e+05,13.184476,0.0,0.073143,10961.27248
min,0.0,1.300000e+01,0.009600,0.0,8.444000,303.00000
25%,0.0,2.395000e+02,18.508650,0.0,8.468000,9744.00000
50%,0.0,4.265000e+02,24.131150,0.0,8.499500,17653.50000
75%,0.0,1.420875e+04,28.511000,0.0,8.541000,26997.75000
max,0.0,1.181678e+06,54.491000,0.0,8.717000,39175.00000


In [32]:

genre_df = df[["id", "genre_ids"]].explode("genre_ids")
print(genre_df)

genre_counts = genre_df["genre_ids"].value_counts()
genre_counts.head()




         id          genre_ids
0       278         "[18, 80]"
1       238         "[18, 80]"
2       240         "[18, 80]"
3       424  "[18, 36, 10752]"
4       389             "[18]"
5   1181678      "[35, 10749]"
6       129  "[16, 10751, 14]"
7       155     "[28, 80, 53]"
8     19404  "[35, 18, 10749]"
9       497     "[14, 18, 80]"
10      122     "[12, 14, 28]"
11   496243     "[35, 53, 18]"
12      680     "[53, 80, 35]"
13   372058  "[16, 10749, 18]"
14   157336    "[12, 18, 878]"
15      429             "[37]"
16       13  "[35, 18, 10749]"
17      769         "[18, 80]"
18      346         "[28, 18]"
19    12477  "[16, 18, 10752]"


,count
genre_ids,
"""[18, 80]""",4
"""[35, 18, 10749]""",2
"""[18]""",1
"""[18, 36, 10752]""",1
"""[35, 10749]""",1


In [25]:
df.isna().sum()

,0
adult,0
backdrop_path,0
genre_ids,0
id,0
original_language,0
original_title,0
overview,0
popularity,0
poster_path,0
release_date,0
